# 22 — Export du modèle pour l'application


## 1. Relire la configuration retenue

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import platform
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
TABLES = ROOT / "data" / "02_Tables_originales"
MANIFESTES = ROOT / "data" / "05_Manifestes"
SORTIE_APP = ROOT / "application" / "modele"
SORTIE_APP.mkdir(parents=True, exist_ok=True)

dossiers = sorted(MANIFESTES.glob("modele_final_21_*"))
if not dossiers:
    raise FileNotFoundError("Aucun export du notebook 21 : lance-le d'abord.")
fiche21 = json.loads((dossiers[-1] / "modele_final.json").read_text(encoding="utf-8"))

K_FINAL = int(fiche21["choix"]["K_visites"])
FORME = fiche21["choix"]["representation"]
MODELE = fiche21["choix"]["modele"]
SEUIL = float(fiche21["test_independant"]["seuil"])

print("Configuration retenue par le notebook 21 :")
print(f"  visites utilisees : {K_FINAL}")
print(f"  representation    : {FORME}")
print(f"  modele            : {MODELE}")
print(f"  seuil d'alerte    : {SEUIL}")
print(f"\nPerformance mesuree sur le test independant :")
print(f"  ROC-AUC = {fiche21['test_independant']['roc_auc']} "
      f"(IC 95 % {fiche21['test_independant']['IC95']})")
print(f"  sensibilite {fiche21['test_independant']['sensibilite']}, "
      f"specificite {fiche21['test_independant']['specificite']}, "
      f"VPN {fiche21['test_independant']['VPN']}")

## 2. Reconstruire la cohorte

Exactement le même code que le notebook 21, pour qu'il n'y ait aucune divergence possible
entre ce qui a été évalué et ce qui est mis en service.

In [ ]:
HORIZON, FENETRE, TOLERANCE_SCORE = 730, (640, 820), 90

def lire(prefixe):
    f = sorted(TABLES.rglob(prefixe + "*.csv"))
    if not f:
        return None
    t = pd.read_csv(f[-1], dtype="string", encoding="utf-8-sig")
    t.columns = t.columns.str.strip().str.upper()
    if "PTID" in t.columns:
        t["PTID"] = t["PTID"].str.strip().str.upper()
    return t

def en_date(s):
    return pd.to_datetime(s.astype("string").str.strip().str.slice(0, 10),
                          format="%Y-%m-%d", errors="coerce").dt.normalize()

dxb = lire("All_Subjects_DXSUM_")
dxb["date_dx"] = en_date(dxb["EXAMDATE"])
dxb["code_dx"] = pd.to_numeric(dxb["DIAGNOSIS"], errors="coerce")
dxb = dxb.loc[dxb["PTID"].str.fullmatch(r"\d{3}_S_\d{4,}", na=False)
              & dxb["date_dx"].notna() & dxb["code_dx"].isin([1, 2, 3]),
              ["PTID", "date_dx", "code_dx"]]
n_codes = dxb.groupby(["PTID", "date_dx"])["code_dx"].nunique()
conflits = n_codes[n_codes > 1].reset_index()[["PTID", "date_dx"]]
dxb = (dxb.loc[~pd.MultiIndex.from_frame(dxb[["PTID", "date_dx"]])
               .isin(pd.MultiIndex.from_frame(conflits))]
       .drop_duplicates().sort_values(["PTID", "date_dx"]))
dxb["code_dx"] = dxb["code_dx"].astype(int)
dxb["rang"] = dxb.groupby("PTID").cumcount()

fixes = pd.DataFrame({"PTID": sorted(dxb["PTID"].unique())})
demo = lire("All_Subjects_PTDEMOG_")
if demo is not None:
    d = demo.copy()
    for brut, nom in [("PTGENDER", "sexe"), ("PTEDUCAT", "education")]:
        if brut in d.columns:
            d[nom] = pd.to_numeric(d[brut], errors="coerce")
    col_dob = next((c for c in ("PTDOBYY", "PTDOB", "PTDOBYEAR") if c in d.columns), None)
    if col_dob:
        d["naissance"] = pd.to_numeric(
            d[col_dob].astype("string").str.extract(r"(\d{4})")[0], errors="coerce")
        d.loc[~d["naissance"].between(1850, 2026), "naissance"] = np.nan
    garde = [c for c in ("sexe", "education", "naissance") if c in d.columns]
    d = (d[["PTID"] + garde].dropna(how="all", subset=garde)
         .drop_duplicates("PTID", keep="first"))
    fixes = fixes.merge(d, on="PTID", how="left", validate="one_to_one")

apoe = lire("All_Subjects_APOERES_")
if apoe is not None and "GENOTYPE" in apoe.columns:
    g = apoe[["PTID", "GENOTYPE"]].copy()
    geno = g["GENOTYPE"].astype("string").str.replace(" ", "", regex=False)
    g["APOE_e4"] = geno.where(geno.str.fullmatch(r"[234]/[234]", na=False)).map(
        lambda s: str(s).count("4") if pd.notna(s) else np.nan)
    g = g.dropna(subset=["APOE_e4"]).drop_duplicates("PTID", keep="first")[["PTID", "APOE_e4"]]
    fixes = fixes.merge(g, on="PTID", how="left", validate="one_to_one")

COLS_FIXES = [c for c in ("sexe", "education", "APOE_e4") if c in fixes.columns]

SCORES = {"All_Subjects_MMSE_": ("VISDATE", {"MMSCORE": "MMSE"}),
          "All_Subjects_CDR_": ("VISDATE", {"CDGLOBAL": "CDR_global", "CDRSB": "CDR_SB"}),
          "All_Subjects_ADAS_": ("VISDATE", {"TOTAL13": "ADAS13"}),
          "All_Subjects_FAQ_": ("VISDATE", {"FAQTOTAL": "FAQ"})}

def table_score(prefixe, col_date, corr):
    t = lire(prefixe)
    if t is None or col_date not in t.columns:
        return None
    t["date_mesure"] = en_date(t[col_date])
    noms = []
    for brut, nom in corr.items():
        if brut in t.columns:
            t[nom] = pd.to_numeric(t[brut], errors="coerce")
            t[nom] = t[nom].where(t[nom] >= 0)   # -1 = valeur manquante dans ADNI, pas un vrai score
            noms.append(nom)
    if not noms:
        return None
    return (t.loc[t["date_mesure"].notna(), ["PTID", "date_mesure"] + noms]
            .dropna(subset=noms, how="all")
            .drop_duplicates(subset=["PTID", "date_mesure"])
            .sort_values("date_mesure"))

def construire(K):
    nb = dxb.groupby("PTID")["rang"].max() + 1
    d = dxb.loc[dxb["PTID"].isin(nb[nb >= K + 1].index)]
    ancre = (d.loc[d["rang"] == K - 1].set_index("PTID")[["date_dx", "code_dx"]]
             .rename(columns={"date_dx": "date_Tk", "code_dx": "dx_Tk"}))
    suite = d.merge(ancre, on="PTID")
    suite["delai"] = (suite["date_dx"] - suite["date_Tk"]).dt.days
    cand = suite.loc[suite["delai"].between(*FENETRE)].copy()
    cand["distance"] = (cand["delai"] - HORIZON).abs()
    cible = (cand.sort_values(["PTID", "distance", "date_dx"]).drop_duplicates("PTID")
             [["PTID", "code_dx"]].rename(columns={"code_dx": "dx_cible"}))
    base = ancre.reset_index().merge(cible, on="PTID", how="inner")
    base = base.loc[base["dx_Tk"].isin([1, 2])].copy()
    base["cible"] = (base["dx_cible"] > base["dx_Tk"]).astype(int)

    panel = (d.loc[d["rang"] < K, ["PTID", "date_dx", "code_dx", "rang"]]
             .loc[lambda t: t["PTID"].isin(set(base["PTID"]))]
             .sort_values(["PTID", "rang"]).copy())
    for prefixe, (cd, corr) in SCORES.items():
        t = table_score(prefixe, cd, corr)
        if t is None:
            continue
        noms = [c for c in t.columns if c not in ("PTID", "date_mesure")]
        lien = pd.merge_asof(panel.sort_values("date_dx"),
                             t.loc[t["PTID"].isin(set(panel["PTID"]))],
                             by="PTID", left_on="date_dx", right_on="date_mesure",
                             direction="backward",
                             tolerance=pd.Timedelta(days=TOLERANCE_SCORE))
        panel = panel.merge(lien[["PTID", "rang"] + noms], on=["PTID", "rang"], how="left")

    var_visite = ["code_dx"] + [c for c in ["MMSE", "CDR_global", "CDR_SB", "ADAS13", "FAQ"]
                                if c in panel.columns]
    ptids = sorted(base["PTID"])
    cube = (panel.set_index(["PTID", "rang"])[var_visite]
            .reindex(pd.MultiIndex.from_product([ptids, range(K)], names=["PTID", "rang"]))
            .to_numpy(dtype=float).reshape(len(ptids), K, len(var_visite)))
    f = base.set_index("PTID").loc[ptids, ["date_Tk"]].join(fixes.set_index("PTID"), how="left")
    if "naissance" in f.columns:
        f["age"] = f["date_Tk"].dt.year - f["naissance"]
        f.loc[~f["age"].between(40, 120), "age"] = np.nan
    cols_f = COLS_FIXES + (["age"] if "age" in f.columns else [])
    F = f[cols_f].to_numpy(dtype=float)
    y = base.set_index("PTID").loc[ptids, "cible"].to_numpy(dtype=int)
    return ptids, cube, F, y, var_visite, cols_f

ptids, cube, F, y, var_visite, cols_f = construire(K_FINAL)
if FORME == "statique":
    X = np.hstack([cube[:, -1, :], F])
    NOMS_VARIABLES = list(var_visite) + list(cols_f)
else:
    pentes = cube[:, -1, :] - cube[:, 0, :]
    X = np.hstack([cube.reshape(len(ptids), -1), pentes, F])
    NOMS_VARIABLES = ([f"{v}_visite{i+1}" for i in range(K_FINAL) for v in var_visite]
                      + [f"pente_{v}" for v in var_visite] + list(cols_f))

assert X.shape[1] == len(NOMS_VARIABLES)
print(f"{len(y)} participants | {X.shape[1]} variables | "
      f"{y.sum()} aggravations ({y.mean():.1%})")
print("Variables, dans l'ordre attendu par l'application :")
print(NOMS_VARIABLES)

## 3. Entraîner sur toutes les données et enregistrer

Le modèle enregistré est une chaîne complète : remplissage des valeurs manquantes par la
médiane, mise à l'échelle, puis régression logistique. L'application n'a donc rien à
préparer — elle transmet les valeurs brutes, et les cases laissées vides sont remplies
automatiquement par la médiane apprise ici.

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def construire_modele(nom):
    if nom == "Random forest":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("clf", RandomForestClassifier(n_estimators=300, random_state=0,
                                                        class_weight="balanced", n_jobs=-1))])
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("ech", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])

if MODELE == "Ensemble":
    modele = {"logreg": construire_modele("Regression logistique").fit(X, y),
              "foret": construire_modele("Random forest").fit(X, y)}
    print("Ensemble entraine : moyenne des deux modeles.")
else:
    modele = construire_modele(MODELE).fit(X, y)
    print(f"Modele entraine : {MODELE}")

# --- Calibration -----------------------------------------------------------------------------
# class_weight="balanced" fait croire au modele que 50 % des patients s'aggravent : les probabilites
# brutes sont gonflees (en moyenne 35 % annonces pour 14 % reels). Le CLASSEMENT des patients est bon
# (c'est ce que mesure l'AUC) mais le pourcentage affiche n'est pas un vrai risque.
# On applique donc une calibration de Platt, apprise sur des predictions hors echantillon
# (validation croisee), qui conserve l'ordre des patients et ramene les pourcentages a la realite.
from sklearn.model_selection import StratifiedKFold

def proba_brute(nom_modele, Xa, ya, Xt):
    if nom_modele == "Ensemble":
        return (construire_modele("Regression logistique").fit(Xa, ya).predict_proba(Xt)[:, 1]
                + construire_modele("Random forest").fit(Xa, ya).predict_proba(Xt)[:, 1]) / 2
    return construire_modele(nom_modele).fit(Xa, ya).predict_proba(Xt)[:, 1]

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

oof = np.zeros(len(y))
for app_, val_ in StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y):
    oof[val_] = proba_brute(MODELE, X[app_], y[app_], X[val_])
platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(oof).reshape(-1, 1), y)
PENTE, ORDONNEE = float(platt.coef_[0, 0]), float(platt.intercept_[0])

def calibrer(p):
    return 1 / (1 + np.exp(-(PENTE * logit(p) + ORDONNEE)))

SEUIL_BRUT = SEUIL
SEUIL = float(calibrer(SEUIL_BRUT))   # meme decision qu'avant, exprimee en vraie probabilite
oof_cal = calibrer(oof)
print(f"Calibration : pente = {PENTE:.3f}, ordonnee = {ORDONNEE:.3f}")
print(f"Risque reel moyen {y.mean():.1%} | probabilite moyenne AVANT {oof.mean():.1%} | APRES {oof_cal.mean():.1%}")
print(f"Seuil d'alerte : {SEUIL_BRUT:.3f} (brut) -> {SEUIL:.3f} (calibre), decision identique")
bornes = np.quantile(oof_cal, [0, .2, .4, .6, .8, 1.0])
CONTROLE_CALIBRATION = []
for a, b in zip(bornes[:-1], bornes[1:]):
    m = (oof_cal >= a) & (oof_cal <= b)
    CONTROLE_CALIBRATION.append({"predit": round(float(oof_cal[m].mean()), 3),
                                 "observe": round(float(y[m].mean()), 3), "n": int(m.sum())})
    print(f"  predit {oof_cal[m].mean():6.1%}  ->  observe {y[m].mean():6.1%}   (n = {m.sum()})")

joblib.dump({"type": MODELE, "modele": modele, "variables": NOMS_VARIABLES,
             "calibration": {"methode": "Platt sur predictions hors echantillon",
                             "pente": PENTE, "ordonnee": ORDONNEE}},
            SORTIE_APP / "modele_final.joblib")

# --- quelques reperes pour l'interface : bornes et medianes observees ---
colonnes = pd.DataFrame(X, columns=NOMS_VARIABLES)
reperes = {c: {"mediane": None if pd.isna(colonnes[c].median()) else round(float(colonnes[c].median()), 2),
               "min": None if pd.isna(colonnes[c].min()) else round(float(colonnes[c].min()), 2),
               "max": None if pd.isna(colonnes[c].max()) else round(float(colonnes[c].max()), 2)}
           for c in NOMS_VARIABLES}

fiche = {
    "nom": "Prediction du risque d'aggravation cognitive a deux ans",
    "date_entrainement": datetime.now().isoformat(timespec="seconds"),
    "configuration": {"visites": K_FINAL, "representation": FORME, "modele": MODELE},
    "variables": NOMS_VARIABLES,
    "seuil_alerte": SEUIL,
    "seuil_alerte_brut": SEUIL_BRUT,
    "calibration": {"methode": "Platt sur predictions hors echantillon", "pente": PENTE,
                    "ordonnee": ORDONNEE, "controle": CONTROLE_CALIBRATION},
    "entrainement": {"participants": int(len(y)), "aggravations": int(y.sum()),
                     "taux": round(float(y.mean()), 3)},
    "performance_test_independant": fiche21["test_independant"],
    "reperes_variables": reperes,
    "cible": "diagnostic a environ deux ans superieur a celui de la visite de reference",
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "scikit_learn": __import__("sklearn").__version__},
}
(SORTIE_APP / "modele_final.json").write_text(
    json.dumps(fiche, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nFichiers ecrits :")
for f in sorted(SORTIE_APP.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} Ko)")

## 4. Vérifier que le modèle enregistré fonctionne

In [ ]:
charge = joblib.load(SORTIE_APP / "modele_final.joblib")

def predire(valeurs):
    """valeurs : dict {nom_variable: nombre ou None}. Renvoie la probabilite."""
    ligne = np.array([[valeurs.get(v, np.nan) if valeurs.get(v) is not None else np.nan
                       for v in charge["variables"]]], dtype=float)
    m = charge["modele"]
    if charge["type"] == "Ensemble":
        brut = float((m["logreg"].predict_proba(ligne)[0, 1]
                      + m["foret"].predict_proba(ligne)[0, 1]) / 2)
    else:
        brut = float(m.predict_proba(ligne)[0, 1])
    cal = charge.get("calibration")
    return float(calibrer(brut)) if cal else brut

exemples = {
    "Profil rassurant (normal, scores dans la norme)":
        {"code_dx": 1, "MMSE": 29, "CDR_global": 0, "CDR_SB": 0, "ADAS13": 8,
         "FAQ": 0, "sexe": 2, "education": 16, "APOE_e4": 0, "age": 68},
    "Profil intermediaire (trouble leger, scores moyens)":
        {"code_dx": 2, "MMSE": 27, "CDR_global": 0.5, "CDR_SB": 1.5, "ADAS13": 16,
         "FAQ": 3, "sexe": 1, "education": 12, "APOE_e4": 1, "age": 75},
    "Profil preoccupant (trouble leger, scores degrades)":
        {"code_dx": 2, "MMSE": 24, "CDR_global": 0.5, "CDR_SB": 3.5, "ADAS13": 26,
         "FAQ": 9, "sexe": 1, "education": 10, "APOE_e4": 2, "age": 79},
    "Donnees partielles (seulement le diagnostic et le MMSE)":
        {"code_dx": 2, "MMSE": 26},
}
lignes = []
for libelle, v in exemples.items():
    p = predire(v)
    lignes.append({"profil": libelle, "probabilite": round(p, 3),
                   "decision": "alerte" if p >= SEUIL else "pas d'alerte"})
display(pd.DataFrame(lignes))

print(f"\nSeuil d'alerte : {SEUIL}")
print("Le dernier exemple montre que le modele fonctionne avec des cases vides :")
print("les valeurs absentes sont remplacees par la mediane apprise a l'entrainement.")
print("\nLe modele est pret. Lance l'application depuis le dossier application\\ :")
print("   python app.py")